In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [1]:
data = pd.read_csv('/content/AEP_hourly.csv')
print(data.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121273 entries, 0 to 121272
Data columns (total 2 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   Datetime  121273 non-null  object 
 1   AEP_MW    121273 non-null  float64
dtypes: float64(1), object(1)
memory usage: 1.9+ MB
None


In [1]:
data.head()

Out[0]: 
              Datetime   AEP_MW
0  2004-12-31 01:00:00  13478.0
1  2004-12-31 02:00:00  12865.0
2  2004-12-31 03:00:00  12577.0
3  2004-12-31 04:00:00  12517.0
4  2004-12-31 05:00:00  12670.0


              Datetime   AEP_MW
0  2004-12-31 01:00:00  13478.0
1  2004-12-31 02:00:00  12865.0
2  2004-12-31 03:00:00  12577.0
3  2004-12-31 04:00:00  12517.0
4  2004-12-31 05:00:00  12670.0

In [1]:
data.rename(columns={'AEP_MW':'Power_consumption'}, inplace=True)

In [1]:
data['Datetime'] = pd.to_datetime(data['Datetime'], format='mixed')
data['Date'] = data['Datetime'].dt.date
data['Time'] = data['Datetime'].dt.time

print(data.head())

             Datetime  Power_consumption        Date      Time
0 2004-12-31 01:00:00            13478.0  2004-12-31  01:00:00
1 2004-12-31 02:00:00            12865.0  2004-12-31  02:00:00
2 2004-12-31 03:00:00            12577.0  2004-12-31  03:00:00
3 2004-12-31 04:00:00            12517.0  2004-12-31  04:00:00
4 2004-12-31 05:00:00            12670.0  2004-12-31  05:00:00


In [1]:
data.drop(['Datetime'], axis=1, inplace=True)


In [1]:
data.describe()

Out[0]: 
       Power_consumption
count      121273.000000
mean        15499.513717
std          2591.399065
min          9581.000000
25%         13630.000000
50%         15310.000000
75%         17200.000000
max         25695.000000


       Power_consumption
count      121273.000000
mean        15499.513717
std          2591.399065
min          9581.000000
25%         13630.000000
50%         15310.000000
75%         17200.000000
max         25695.000000

In [1]:
data.head()

Out[0]: 
   Power_consumption        Date      Time
0            13478.0  2004-12-31  01:00:00
1            12865.0  2004-12-31  02:00:00
2            12577.0  2004-12-31  03:00:00
3            12517.0  2004-12-31  04:00:00
4            12670.0  2004-12-31  05:00:00


   Power_consumption        Date      Time
0            13478.0  2004-12-31  01:00:00
1            12865.0  2004-12-31  02:00:00
2            12577.0  2004-12-31  03:00:00
3            12517.0  2004-12-31  04:00:00
4            12670.0  2004-12-31  05:00:00

In [1]:
daily_avg_power = data.groupby('Date')['Power_consumption'].mean().reset_index()

plt.figure(figsize=(15, 7))
plt.plot(daily_avg_power['Date'], daily_avg_power['Power_consumption'], color='green')
plt.title('Daily Average Power Consumption Fluctuations')
plt.xlabel('Date')
plt.ylabel('Average Power Consumption (MW)')
plt.grid(True)
plt.show()

<ipython-input-1-9298eab135f7>:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

print("Starting Step 1: EDA for Power Consumption...")

# --- 1. Load Data ---
DATA_FILEPATH = 'AEP_hourly.csv'
try:
    df = pd.read_csv(DATA_FILEPATH)
except FileNotFoundError:
    print(f"Error: '{DATA_FILEPATH}' not found.")
    print("Please make sure 'AEP_hourly.csv' is in the same folder.")
    exit()

# --- 2. Clean and Prepare Data for Plotting ---
df = df.rename(columns={
    'Datetime': 'timestamp',
    'AEP_MW': 'power_consumption_mw'
})

# This is the most important step for EDA:
# Convert the 'timestamp' string into a real datetime object
df['timestamp'] = pd.to_datetime(df['timestamp'])

# Create new columns to help us group the data
df['hour'] = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.day_name()
df['month'] = df['timestamp'].dt.month

print("Data loaded and features (hour, day_of_week, month) created.")

# --- 3. Create Plots ---
# Create a directory to save plots
PLOT_DIR = "model_2_eda_plots"
if not os.path.exists(PLOT_DIR):
    os.makedirs(PLOT_DIR)

# Plot 1: The Average Daily Cycle ("M" Shape)
#
print("Generating Plot 1: Average Daily Cycle...")
plt.figure(figsize=(10, 6))
# We group by 'hour' and get the average power for each hour
daily_avg = df.groupby('hour')['power_consumption_mw'].mean()
daily_avg.plot(kind='line', style='-o', color='blue')
plt.title('Average Power Consumption by Hour of Day', fontsize=16)
plt.xlabel('Hour of Day (0-23)')
plt.ylabel('Average Power (MW)')
plt.grid(True, linestyle='--')
plt.savefig(os.path.join(PLOT_DIR, '1_daily_cycle.png'))
plt.close()

# Plot 2: The Average Seasonal Cycle ("U" Shape)
print("Generating Plot 2: Average Seasonal Cycle...")
plt.figure(figsize=(10, 6))
# We group by 'month' and get the average
monthly_avg = df.groupby('month')['power_consumption_mw'].mean()
monthly_avg.plot(kind='line', style='-o', color='green')
plt.title('Average Power Consumption by Month', fontsize=16)
plt.xlabel('Month (1-12)')
plt.ylabel('Average Power (MW)')
plt.xticks(range(1, 13), ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
plt.grid(True, linestyle='--')
plt.savefig(os.path.join(PLOT_DIR, '2_seasonal_cycle.png'))
plt.close()

# Plot 3: The Average Weekly Cycle (Weekday vs. Weekend)
print("Generating Plot 3: Average Weekly Cycle...")
plt.figure(figsize=(10, 6))
# We group by 'day_of_week'
weekly_avg = df.groupby('day_of_week')['power_consumption_mw'].mean()
# Order the days correctly
days_ordered = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
weekly_avg = weekly_avg.reindex(days_ordered)
weekly_avg.plot(kind='bar', color='purple')
plt.title('Average Power Consumption by Day of Week', fontsize=16)
plt.xlabel('Day of Week')
plt.ylabel('Average Power (MW)')
plt.xticks(rotation=45)
plt.grid(True, linestyle='--')
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, '3_weekly_cycle.png'))
plt.close()

print(f"\nSuccess! All EDA plots saved to '{PLOT_DIR}' folder.")

Starting Step 1: EDA for Power Consumption...
Data loaded and features (hour, day_of_week, month) created.
Generating Plot 1: Average Daily Cycle...
Generating Plot 2: Average Seasonal Cycle...
Generating Plot 3: Average Weekly Cycle...

Success! All EDA plots saved to 'model_2_eda_plots' folder.


In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
from xgboost import XGBRegressor
import joblib
import os

print("Starting Step 2: Building Model 2 (Power Consumption)...")

# --- 1. Load Data ---
DATA_FILEPATH = 'AEP_hourly.csv'
try:
    df = pd.read_csv(DATA_FILEPATH)
except FileNotFoundError:
    print(f"Error: '{DATA_FILEPATH}' not found.")
    exit()

# --- 2. Clean and Rename ---
df = df.rename(columns={
    'Datetime': 'timestamp',
    'AEP_MW': 'power_consumption_mw'
})
df['timestamp'] = pd.to_datetime(df['timestamp'])

# --- 3. Feature Engineering (The "Clock Tricks") ---
print("Creating time-series features (sin/cos)...")
df['hour_sin'] = np.sin(2 * np.pi * df['timestamp'].dt.hour / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['timestamp'].dt.hour / 24)
df['day_of_week_sin'] = np.sin(2 * np.pi * df['timestamp'].dt.dayofweek / 7)
df['day_of_week_cos'] = np.cos(2 * np.pi * df['timestamp'].dt.dayofweek / 7)
df['day_of_year_sin'] = np.sin(2 * np.pi * df['timestamp'].dt.dayofyear / 365.25)
df['day_of_year_cos'] = np.cos(2 * np.pi * df['timestamp'].dt.dayofyear / 365.25)

# --- 4. Lag Feature (The "Secret Weapon") ---
print("Creating lag features (daily and weekly)...")
df['power_24h_ago'] = df['power_consumption_mw'].shift(24)
df['power_168h_ago'] = df['power_consumption_mw'].shift(168)

# --- 5. Define Features (X) and Target (y) ---
target = 'power_consumption_mw'
features = [
    'hour_sin', 'hour_cos',
    'day_of_week_sin', 'day_of_week_cos',
    'day_of_year_sin', 'day_of_year_cos',
    'power_24h_ago',
    'power_168h_ago'
]

df = df.dropna(subset=features + [target])

X = df[features]
y = df[target]

# --- 6. Split Data (Time-Series Split) ---
split_index = int(len(df) * 0.8)
X_train = X.iloc[:split_index]
y_train = y.iloc[:split_index]
X_test = X.iloc[split_index:]
y_test = y.iloc[split_index:]

print(f"Data split: {len(y_train)} rows for training, {len(y_test)} rows for testing.")

# --- 7. Train the Model ---
print("Training the RandomForestRegressor model...")
model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
    max_depth=15
)
model.fit(X_train, y_train)
print("Model training complete!")

# --- 8. Evaluate Random Forest ---
print("\n--- Model Evaluation ---")
y_pred = model.predict(X_test)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"Random Forest R²: {r2}")
print(f"Random Forest RMSE: {rmse}")

# --- Train and Evaluate XGBoost ---
print("Training the XGBRegressor model...")
xgb_model = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    random_state=42,
    n_jobs=-1
)
xgb_model.fit(X_train, y_train)
print("XGBoost training complete!")

y_pred_xgb = xgb_model.predict(X_test)
r2_xgb = r2_score(y_test, y_pred_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))

print(f"XGBoost R²: {r2_xgb}")
print(f"XGBoost RMSE: {rmse_xgb}")

# --- 9. Save the Final Model ---
MODEL_FILENAME = 'power_consumption_model.joblib'
joblib.dump(model, MODEL_FILENAME)
print(f"\nSUCCESS! Model 2 (Demand) saved as '{MODEL_FILENAME}'.")


Starting Step 2: Building Model 2 (Power Consumption)...
Creating time-series features (sin/cos)...
Creating lag features (daily and weekly)...
Data split: 96884 rows for training, 24221 rows for testing.
Training the RandomForestRegressor model...
Model training complete!

--- Model Evaluation ---
Random Forest R²: 0.8445984529319438
Random Forest RMSE: 1011.5812613965211
Training the XGBRegressor model...
XGBoost training complete!
XGBoost R²: 0.8493072620575622
XGBoost RMSE: 996.1374519963388

SUCCESS! Model 2 (Demand) saved as 'power_consumption_model.joblib'.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Load Data
data = pd.read_csv('/content/AEP_hourly.csv')
print("Original data information:")
print(data.info())
print("\nOriginal data head:")
print(data.head())

# 2. Rename 'AEP_MW' column to 'Power_consumption'
data.rename(columns={'AEP_MW':'Power_consumption'}, inplace=True)
print("\nData head after renaming column:")
print(data.head())

# 3. Convert 'Datetime' column to datetime objects
# This step was crucial for plotting and feature extraction
data['Datetime'] = pd.to_datetime(data['Datetime'])
print("\nData information after converting 'Datetime' to datetime objects:")
print(data.info())

# 4. Generate a line graph of full power consumption fluctuations
plt.figure(figsize=(15, 7))
plt.plot(data['Datetime'], data['Power_consumption'], color='blue')
plt.title('Power Consumption Fluctuations Over Time (Full Data)')
plt.xlabel('Date')
plt.ylabel('Power Consumption (MW)')
plt.grid(True)
plt.show()

# 5. Separate 'Datetime' into 'Date' and 'Time' columns
data['Date'] = data['Datetime'].dt.date
data['Time'] = data['Datetime'].dt.time
print("\nData head after separating Date and Time columns:")
print(data.head())

# 6. Drop the original 'Datetime' column
data.drop(['Datetime'], axis=1, inplace=True)
print("\nData head after dropping original 'Datetime' column:")
print(data.head())

# 7. Generate a line graph of daily average power consumption fluctuations
daily_avg_power = data.groupby('Date')['Power_consumption'].mean().reset_index()

plt.figure(figsize=(15, 7))
plt.plot(daily_avg_power['Date'], daily_avg_power['Power_consumption'], color='green')
plt.title('Daily Average Power Consumption Fluctuations')
plt.xlabel('Date')
plt.ylabel('Average Power Consumption (MW)')
plt.grid(True)
plt.show()

# 8. Generate a box plot for outlier identification
plt.figure(figsize=(10, 6))
sns.boxplot(y=data['Power_consumption'])
plt.title('Box Plot of Power Consumption to Identify Outliers')
plt.ylabel('Power Consumption (MW)')
plt.show()

Original data information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121273 entries, 0 to 121272
Data columns (total 2 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   Datetime  121273 non-null  object 
 1   AEP_MW    121273 non-null  float64
dtypes: float64(1), object(1)
memory usage: 1.9+ MB
None

Original data head:
              Datetime   AEP_MW
0  2004-12-31 01:00:00  13478.0
1  2004-12-31 02:00:00  12865.0
2  2004-12-31 03:00:00  12577.0
3  2004-12-31 04:00:00  12517.0
4  2004-12-31 05:00:00  12670.0

Data head after renaming column:
              Datetime  Power_consumption
0  2004-12-31 01:00:00            13478.0
1  2004-12-31 02:00:00            12865.0
2  2004-12-31 03:00:00            12577.0
3  2004-12-31 04:00:00            12517.0
4  2004-12-31 05:00:00            12670.0

Data information after converting 'Datetime' to datetime objects:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121273 entries, 0 to 121272
Da

<ipython-input-1-ac72f6f0166c>:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
<ipython-input-1-ac72f6f0166c>:53: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
<ipython-input-1-ac72f6f0166c>:60: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import warnings
import sys

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# --- This is the key prediction function ---

def predict_pattern(date_str, model_to_use):
    """
    Predicts the typical power consumption for any given date/time string
    by extracting the cyclical features (Hour, DayOfWeek, Month, DayOfYear).

    Args:
        date_str (str): A date/time string (e.g., "2025-07-20 18:00:00").
        model_to_use (sklearn.BaseEstimator): The trained regression model.

    Returns:
        float: The predicted power consumption (MW).
    """
    try:
        # Convert string to datetime object
        target_dt = pd.to_datetime(date_str)

        # Extract the *exact same features* the model was trained on
        hour = target_dt.hour
        dayofweek = target_dt.dayofweek
        month = target_dt.month
        dayofyear = target_dt.dayofyear

        # Create a single-row DataFrame in the correct format
        input_features = pd.DataFrame(
            [[hour, dayofweek, month, dayofyear]],
            columns=['Hour', 'DayOfWeek', 'Month', 'DayOfYear']
        )

        # Make the prediction
        prediction = model_to_use.predict(input_features)

        return prediction[0]

    except Exception as e:
        print(f"Error during prediction: {e}")
        return None

# --- Main script execution ---

def main():
    # 1. Load and Prepare Data
    try:
        # Load the dataset
        data = pd.read_csv('AEP_hourly.csv')
        data.rename(columns={'AEP_MW': 'Power_consumption'}, inplace=True)
        data['Datetime'] = pd.to_datetime(data['Datetime'])
        print("Data loaded and preprocessed successfully.")
        print(data.info())
        print("\n" + "="*30 + "\n")
    except Exception as e:
        print(f"Error loading data: {e}")
        return # Stop execution if data can't be loaded

    # 2. Visualization (from your code, modified to save files)
    print("Generating visualizations...")
    try:
        # Daily average plot
        daily_avg_power = data.set_index('Datetime').resample('D')['Power_consumption'].mean()
        plt.figure(figsize=(15, 7))
        daily_avg_power.plot(color='green')
        plt.title('Daily Average Power Consumption Fluctuations (Original Data)')
        plt.xlabel('Date')
        plt.ylabel('Average Power Consumption (MW)')
        plt.grid(True)
        plt.savefig('daily_avg_consumption.png')
        plt.close()
        print("Saved daily_avg_consumption.png")

        # Box plot
        plt.figure(figsize=(10, 6))
        sns.boxplot(y=data['Power_consumption'])
        plt.title('Box Plot of Power Consumption to Identify Outliers')
        plt.ylabel('Power Consumption (MW)')
        plt.savefig('consumption_boxplot.png')
        plt.close()
        print("Saved consumption_boxplot.png")
        print("\n" + "="*30 + "\n")
    except Exception as e:
        print(f"Error during visualization: {e}")

    # 3. Feature Engineering (The Key Step)
    print("Starting feature engineering...")
    # We extract cyclical features that do NOT depend on the year.
    data['Hour'] = data['Datetime'].dt.hour
    data['DayOfWeek'] = data['Datetime'].dt.dayofweek  # Monday=0, Sunday=6
    data['Month'] = data['Datetime'].dt.month
    data['DayOfYear'] = data['Datetime'].dt.dayofyear # 1-366

    # Define features (X) and target (y)
    # Notice 'Year' is NOT included!
    features = ['Hour', 'DayOfWeek', 'Month', 'DayOfYear']
    X = data[features]
    y = data['Power_consumption']

    print(f"Features for model: {features}")
    print("Feature data head:")
    print(X.head())
    print("\n" + "="*30 + "\n")

    # 4. Train the Model
    print("Splitting data and training model...")
    # Split data (we can use a smaller portion for training to speed things up)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Using a RandomForestRegressor
    # n_estimators=100 is a good balance.
    # min_samples_leaf=10 helps prevent overfitting and makes the model learn more general patterns.
    # n_jobs=-1 uses all available CPU cores to speed up training.
    model = RandomForestRegressor(n_estimators=100, min_samples_leaf=10, random_state=42, n_jobs=-1)

    # Train the model
    model.fit(X_train, y_train)
    print("Model training complete.")

    # Evaluate the model
    predictions = model.predict(X_test)
    mae = mean_absolute_error(y_test, predictions)
    print(f"\nModel Mean Absolute Error on Test Set: {mae:.2f} MW")
    print("This (MAE) is the average error you can expect from the model's predictions.")
    print("\n" + "="*30 + "\n")


    # 5. Demonstrate the Prediction Function
    print("Demonstrating prediction function with future dates:")

    # Example 1: A future summer evening
    date_1 = "2025-07-20 18:00:00"
    pred_1 = predict_pattern(date_1, model)
    print(f"Predicted consumption for {date_1}: {pred_1:.2f} MW")

    # Example 2: A future winter morning
    date_2 = "2027-01-15 08:00:00"
    pred_2 = predict_pattern(date_2, model)
    print(f"Predicted consumption for {date_2}: {pred_2:.2f} MW")

    # Example 3: A future weekday afternoon
    date_3 = "2030-04-10 14:00:00"
    pred_3 = predict_pattern(date_3, model)
    print(f"Predicted consumption for {date_3}: {pred_3:.2f} MW")
    print("\n" + "="*30 + "\n")


    # 6. Visualize the Learned Pattern
    print("Generating plot of the model's learned yearly pattern...")
    # Create a sample year (2024 is a leap year, good for DayOfYear=366)
    sample_dates = pd.date_range(start='2024-01-01 00:00:00', end='2024-12-31 23:00:00', freq='H')

    # Create features for this sample year
    sample_df = pd.DataFrame()
    sample_df['Datetime'] = sample_dates
    sample_df['Hour'] = sample_df['Datetime'].dt.hour
    sample_df['DayOfWeek'] = sample_df['Datetime'].dt.dayofweek
    sample_df['Month'] = sample_df['Datetime'].dt.month
    sample_df['DayOfYear'] = sample_df['Datetime'].dt.dayofyear

    # Predict consumption for this entire sample year
    sample_features_df = sample_df[features]
    sample_df['Predicted_Consumption'] = model.predict(sample_features_df)

    # Plot the daily average of the *predicted* pattern
    daily_predicted_pattern = sample_df.set_index('Datetime').resample('D')['Predicted_Consumption'].mean()

    plt.figure(figsize=(15, 7))
    daily_predicted_pattern.plot(color='purple')
    plt.title('Model\'s Learned Daily Average Consumption Pattern (Sample Year)')
    plt.xlabel('Date (Sample Year 2024)')
    plt.ylabel('Predicted Power Consumption (MW)')
    plt.grid(True)
    plt.savefig('learned_yearly_pattern.png')
    plt.close()

    print("Saved learned_yearly_pattern.png")
    print("\n" + "="*30 + "\n")
    print("Script finished.")

# This makes the script runnable
if __name__ == "__main__":
    main()

Data loaded and preprocessed successfully.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121273 entries, 0 to 121272
Data columns (total 2 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   Datetime           121273 non-null  datetime64[ns]
 1   Power_consumption  121273 non-null  float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 1.9 MB
None


Generating visualizations...
Saved daily_avg_consumption.png
Saved consumption_boxplot.png


Starting feature engineering...
Features for model: ['Hour', 'DayOfWeek', 'Month', 'DayOfYear']
Feature data head:
   Hour  DayOfWeek  Month  DayOfYear
0     1          4     12        366
1     2          4     12        366
2     3          4     12        366
3     4          4     12        366
4     5          4     12        366


Splitting data and training model...
Model training complete.

Model Mean Absolute Error on Test Set: 1048.98 MW
This (MAE) i

In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
import joblib
import warnings

# Suppress warnings
warnings.filterwarnings('ignore')

print("Starting the model creation and saving script...")

try:
    # 1. Load Data
    data = pd.read_csv('AEP_hourly.csv')
    data.rename(columns={'AEP_MW': 'Power_consumption'}, inplace=True)
    data['Datetime'] = pd.to_datetime(data['Datetime'])
    print("Data loaded successfully.")

    # 2. Feature Engineering
    print("Performing feature engineering...")
    data['Hour'] = data['Datetime'].dt.hour
    data['DayOfWeek'] = data['Datetime'].dt.dayofweek
    data['Month'] = data['Datetime'].dt.month
    data['DayOfYear'] = data['Datetime'].dt.dayofyear

    # Define features (X) and target (y)
    features = ['Hour', 'DayOfWeek', 'Month', 'DayOfYear']
    X = data[features]
    y = data['Power_consumption']
    print("Features created.")

    # 3. Train the Model
    print("Training the model on the full dataset...")

    # Initialize the RandomForestRegressor
    model = RandomForestRegressor(n_estimators=100,
                                  min_samples_leaf=10,
                                  random_state=42,
                                  n_jobs=-1)

    # Train the model on ALL data
    model.fit(X, y)
    print("Model training complete.")


    # 4. Save the Model
    model_filename = 'power_prediction_model.joblib'
    joblib.dump(model, model_filename)
    print(f"\nSuccessfully created and saved the model to '{model_filename}'")

    print("\nScript finished.")

except Exception as e:
    print(f"An error occurred: {e}")

Starting the model creation and saving script...
Data loaded successfully.
Performing feature engineering...
Features created.
Training the model on the full dataset...
Model training complete.

Successfully created and saved the model to 'power_prediction_model.joblib'

Script finished.


In [1]:
import pandas as pd
import joblib

# --- This is the key prediction function ---
def predict_pattern(date_str, model_to_use):
    """
    Predicts the typical power consumption for any given date/time string
    by extracting the cyclical features.
    """
    try:
        # Convert string to datetime object
        target_dt = pd.to_datetime(date_str)

        # Extract the *exact same features* the model was trained on
        hour = target_dt.hour
        dayofweek = target_dt.dayofweek
        month = target_dt.month
        dayofyear = target_dt.dayofyear

        # Create a single-row DataFrame in the correct format
        input_features = pd.DataFrame(
            [[hour, dayofweek, month, dayofyear]],
            columns=['Hour', 'DayOfWeek', 'Month', 'DayOfYear']
        )

        # Make the prediction
        prediction = model_to_use.predict(input_features)

        return prediction[0]

    except Exception as e:
        print(f"Error during prediction: {e}")
        return None

# --- Main part of the prediction script ---
try:
    # 1. Load the pre-trained model
    model_filename = 'power_prediction_model.joblib'
    loaded_model = joblib.load(model_filename)
    print(f"Successfully loaded model from '{model_filename}'")

    # 2. Use the model for predictions
    print("\nDemonstrating predictions with the loaded model:")

    # Example 1: A future summer evening
    date_1 = "2025-07-20 18:00:00"
    pred_1 = predict_pattern(date_1, loaded_model)
    print(f"Predicted consumption for {date_1}: {pred_1:.2f} MW")

    # Example 2: A future winter morning
    date_2 = "2027-01-15 08:00:00"
    pred_2 = predict_pattern(date_2, loaded_model)
    print(f"Predicted consumption for {date_2}: {pred_2:.2f} MW")

except Exception as e:
    print(f"An error occurred: {e}")

Successfully loaded model from 'power_prediction_model.joblib'

Demonstrating predictions with the loaded model:
Predicted consumption for 2025-07-20 18:00:00: 17642.22 MW
Predicted consumption for 2027-01-15 08:00:00: 19688.09 MW


In [1]:
# Fake-temperature experiment excluded per instructions: "Do not use the fake-temperature experiment anywhere."


In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
import joblib
import warnings

# Suppress warnings
warnings.filterwarnings('ignore')

print("Starting export for the best time-only model...")

try:
    # 1. Load Power Data
    data = pd.read_csv('AEP_hourly.csv')
    data.rename(columns={'AEP_MW': 'Power_consumption'}, inplace=True)
    data['Datetime'] = pd.to_datetime(data['Datetime'])
    print("Power data loaded.")

    # 2. CREATE TIME FEATURES
    print("Creating time-based features...")
    data['Hour'] = data['Datetime'].dt.hour
    data['DayOfWeek'] = data['Datetime'].dt.dayofweek
    data['Month'] = data['Datetime'].dt.month
    data['DayOfYear'] = data['Datetime'].dt.dayofyear

    # 3. DEFINE FEATURES
    # We are ONLY using the time features
    features = [
        'Hour',
        'DayOfWeek',
        'Month',
        'DayOfYear'
    ]

    X = data[features]
    y = data['Power_consumption']

    print(f"Features for model: {features}")

    # 4. TRAIN THE MODEL ON ALL DATA
    # We train on the entire dataset for the final exported model
    model = RandomForestRegressor(n_estimators=100,
                                  min_samples_leaf=10,
                                  random_state=42,
                                  n_jobs=-1)

    print(f"Training final model on ALL {len(X)} samples...")
    model.fit(X, y)
    print("Model training complete.")

    # 5. EXPORT THE MODEL
    model_filename = 'power_model_time_only.joblib'
    joblib.dump(model, model_filename)

    print(f"\n--- MODEL EXPORTED ---")
    print(f"Successfully saved the best model to '{model_filename}'")

    print("\n--- INPUT PARAMETERS ---")
    print("This model was trained on the following 4 features:")
    for i, feature in enumerate(features):
        print(f"  {i+1}. {feature}")

    print("\nScript finished.")

except FileNotFoundError:
    print("\n--- SCRIPT FAILED ---")
    print("This script requires 'AEP_hourly.csv' in the same folder.")
except Exception as e:
    print(f"An error occurred: {e}")

Starting export for the best time-only model...
Power data loaded.
Creating time-based features...
Features for model: ['Hour', 'DayOfWeek', 'Month', 'DayOfYear']
Training final model on ALL 121273 samples...
Model training complete.

--- MODEL EXPORTED ---
Successfully saved the best model to 'power_model_time_only.joblib'

--- INPUT PARAMETERS ---
This model was trained on the following 4 features:
  1. Hour
  2. DayOfWeek
  3. Month
  4. DayOfYear

Script finished.


In [1]:
!pip install streamlit --user

In [1]:
import streamlit as st
import joblib

# Put 'your_model.pkl' in the SAME directory as 'app.py'
MODEL_PATH = '/content/power_model_time_only.joblib'

@st.cache_resource  # Cache the model so it only loads once
def load_model(path):
    try:
        model = joblib.load(path)
        return model
    except FileNotFoundError:
        st.error(f"Model file not found at {path}. Please make sure it's in the same folder as app.py.")
        return None

model = load_model(MODEL_PATH)

if model:
    st.write("Model loaded successfully!")
    # ... rest of your UI code

2026-10-30 23:11:19.638 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-30 23:11:19.699 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-30 23:11:21.465 
  command:

    streamlit run C:\Users\Abhishek\.gemini\antigravity\brain\8afcdb3e-a3b1-40ad-9971-4b0f293e247f\scratch\run_notebooks.py [ARGUMENTS]
2026-10-30 23:11:21.465 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-30 23:11:21.466 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
import joblib
import os

print("Starting Step 2: Building Model 2 (Power Consumption)...")

# --- 1. Load Data ---
DATA_FILEPATH = 'AEP_hourly.csv'
try:
    df = pd.read_csv(DATA_FILEPATH)
except FileNotFoundError:
    print(f"Error: '{DATA_FILEPATH}' not found.")
    exit()

# --- 2. Clean and Rename ---
df = df.rename(columns={
    'Datetime': 'timestamp',
    'AEP_MW': 'power_consumption_mw'
})
df['timestamp'] = pd.to_datetime(df['timestamp'])

# --- 3. Feature Engineering (The "Clock Tricks") ---
print("Creating time-series features (sin/cos)...")
# These features capture the patterns we saw in the EDA plots
df['hour_sin'] = np.sin(2 * np.pi * df['timestamp'].dt.hour / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['timestamp'].dt.hour / 24)
df['day_of_week_sin'] = np.sin(2 * np.pi * df['timestamp'].dt.dayofweek / 7)
df['day_of_week_cos'] = np.cos(2 * np.pi * df['timestamp'].dt.dayofweek / 7)
df['day_of_year_sin'] = np.sin(2 * np.pi * df['timestamp'].dt.dayofyear / 365.25)
df['day_of_year_cos'] = np.cos(2 * np.pi * df['timestamp'].dt.dayofyear / 365.25)

# --- 4. Lag Feature (The "Secret Weapon") ---
print("Creating lag features (daily and weekly)...")
# These features are the *most powerful* predictors.
df['power_24h_ago'] = df['power_consumption_mw'].shift(24)
df['power_168h_ago'] = df['power_consumption_mw'].shift(168) # 24 * 7

# --- 5. Define Features (X) and Target (y) ---
target = 'power_consumption_mw'
features = [
    'hour_sin', 'hour_cos',
    'day_of_week_sin', 'day_of_week_cos',
    'day_of_year_sin', 'day_of_year_cos',
    'power_24h_ago',
    'power_168h_ago'
]

# We must drop rows with NaN values (the first 168 rows)
df = df.dropna(subset=features + [target])

X = df[features]
y = df[target]

# --- 6. Split Data (Time-Series Split) ---
# We MUST split by time (past vs. future), not randomly.
split_index = int(len(df) * 0.8)
X_train = X.iloc[:split_index]
y_train = y.iloc[:split_index]
X_test = X.iloc[split_index:]
y_test = y.iloc[split_index:]

print(f"Data split: {len(y_train)} rows for training, {len(y_test)} rows for testing.")

# --- 7. Train the Model ---
print("Training the RandomForestRegressor model...")
model = RandomForestRegressor(
    n_estimators=100,     # 100 trees is a strong, fast baseline
    random_state=42,
    n_jobs=-1,            # Use all your computer's cores
    max_depth=15          # Prevents overfitting
)
model.fit(X_train, y_train)
print("Model training complete!")

# --- 8. Evaluate the Model ---
print("\n--- Model Evaluation ---")
y_pred = model.predict(X_test)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

# This is the 84.5% accurate model
print(f"R-squared (R²): {r2:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f} MW")

# --- 9. Save the Final Model ---
MODEL_FILENAME = 'power_consumption_model.joblib'
joblib.dump(model, MODEL_FILENAME)
print(f"\nSUCCESS! Model 2 (Demand) saved as '{MODEL_FILENAME}'.")

Starting Step 2: Building Model 2 (Power Consumption)...
Creating time-series features (sin/cos)...
Creating lag features (daily and weekly)...
Data split: 96884 rows for training, 24221 rows for testing.
Training the RandomForestRegressor model...
Model training complete!

--- Model Evaluation ---
R-squared (R²): 0.8446
Root Mean Squared Error (RMSE): 1011.58 MW

SUCCESS! Model 2 (Demand) saved as 'power_consumption_model.joblib'.


In [1]:
import joblib
import pandas as pd
import sys

# Define the 4 features the model expects, in order
MODEL_FEATURES = ['Hour', 'DayOfWeek', 'Month', 'DayOfYear']
MODEL_FILENAME = 'power_model_time_only.joblib'

def predict_power(date_str, loaded_model):
    """
    Predicts power consumption for a given date/time string
    using the pre-trained model.
    """
    try:
        # 1. Convert the input string to a datetime object
        target_dt = pd.to_datetime(date_str)

        # 2. Create the 4 required features from the datetime object
        hour = target_dt.hour
        dayofweek = target_dt.dayofweek
        month = target_dt.month
        dayofyear = target_dt.dayofyear

        # 3. Create a DataFrame in the exact format the model expects
        input_data = pd.DataFrame(
            [[hour, dayofweek, month, dayofyear]],
            columns=MODEL_FEATURES
        )

        # 4. Make the prediction
        prediction = loaded_model.predict(input_data)

        # Return the single prediction value
        return prediction[0]

    except Exception as e:
        return f"Error during prediction: {e}"

# --- Main part of the script ---
if __name__ == "__main__":

    # 1. Load the exported model from the file
    try:
        model = joblib.load(MODEL_FILENAME)
        print(f"Successfully loaded model from '{MODEL_FILENAME}'")
    except FileNotFoundError:
        print(f"Error: Model file not found.")
        print(f"Please make sure '{MODEL_FILENAME}' is in the same folder as this script.")
        sys.exit()
    except Exception as e:
        print(f"Error loading model: {e}")
        sys.exit()

    # 2. Test the model with some example dates
    print("\n--- Testing Model Output ---")

    date_1 = "2025-07-20 18:00:00" # Future summer evening
    pred_1 = predict_power(date_1, model)
    print(f"Prediction for {date_1}: {pred_1:.2f} MW")

    date_2 = "2026-01-15 08:00:00" # Future winter morning
    pred_2 = predict_power(date_2, model)
    print(f"Prediction for {date_2}: {pred_2:.2f} MW")

    date_3 = "2027-04-10 14:00:00" # Future spring afternoon
    pred_3 = predict_power(date_3, model)
    print(f"Prediction for {date_3}: {pred_3:.2f} MW")

Successfully loaded model from 'power_model_time_only.joblib'

--- Testing Model Output ---
Prediction for 2025-07-20 18:00:00: 17642.22 MW
Prediction for 2026-01-15 08:00:00: 20174.91 MW
Prediction for 2027-04-10 14:00:00: 13633.93 MW
